# 🚀 AnyProjector — Phase 3 LoRA Tool-Calling (128T + Encoder Unfreeze)

Train the LLM (LoRA) to read 128 audio tokens and generate tool-calling responses.

**Pipeline:** Audio → Whisper Encoder (last 1 layer unfrozen) → Q-Former 128T (frozen) → Qwen2.5-1.5B (LoRA) → Loss

> Phase 2 aligned projector output to text embedding space (cos_sim ≈ 0.75).  
> Phase 3 teaches the LLM to **decode** those audio tokens into tool calls or natural responses.


In [ ]:
# Install dependencies
!pip install -q transformers datasets torch accelerate hf_transfer huggingface_hub peft bitsandbytes

In [ ]:
# Verify GPU
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('⚠️ No GPU! Go to Runtime → Change runtime type → GPU')
print(f'PyTorch: {torch.__version__}')

In [ ]:
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import login
login()

## ⚙️ Configuration
Edit these settings before running.

In [ ]:
# ============================================
# 🔧 EDIT THESE SETTINGS
# ============================================
ENCODER_ID     = 'openai/whisper-medium'
LLM_ID         = 'Qwen/Qwen2.5-1.5B-Instruct'
DATASET_ID     = 'Niem/speech-massive-vie-tool-calling'

# Projector checkpoint (128 tokens, Phase 2)
PROJECTOR_CKPT = '/content/drive/MyDrive/AnyProjector/checkpoints/phase2/v097_antiplateau_128/projector_final.pt'

# LoRA
LORA_RANK      = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05
LORA_TARGETS   = ['q_proj', 'v_proj']

# Encoder unfreeze
UNFREEZE_LAYERS = 1      # Last N layers unfrozen (~10M params/layer)
ENCODER_LR      = 1e-5   # 20x smaller than LoRA LR

# Training
NUM_EPOCHS     = 30
BATCH_SIZE     = 4       # L4: 4, A100: 8-12
LR             = 2e-4
GRAD_ACCUM     = 4       # Effective batch = BATCH_SIZE * GRAD_ACCUM = 16
WARMUP_RATIO   = 0.05
PATIENCE       = 5
VAL_SPLIT      = 0.2
MAX_TEXT_TOKENS = 256
MAX_GRAD_NORM  = 1.0
SAVE_EVERY     = 5       # Save periodic checkpoint

SAVE_DIR       = 'checkpoints/phase3/lora_128t_enc1'
SAMPLE_RATE    = 16000
NUM_WORKERS    = 2
RESUME_FROM    = None    # Set to checkpoint dir to resume

## 📁 Mount Google Drive
Mount Drive để auto-backup checkpoint + load projector.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BACKUP_DIR = '/content/drive/MyDrive/AnyProjector/checkpoints/phase3/lora_128t_enc1'
os.makedirs(BACKUP_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ Backup folder ready: {BACKUP_DIR}')

## 📊 Load Dataset

In [ ]:
from datasets import load_dataset

print(f'Loading {DATASET_ID}...')
raw_ds = load_dataset(DATASET_ID)
print(f'Splits: {list(raw_ds.keys())}')

first_split = list(raw_ds.keys())[0]
print(f'Columns: {raw_ds[first_split].column_names}')
print(f'Total: {sum(len(raw_ds[s]) for s in raw_ds)}')

sample = raw_ds[first_split][0]
print(f'\nSample keys: {[k for k in sample if k != "audio"]}')
print(f'Instruction: {str(sample.get("instruction", ""))[:100]}')
print(f'Output: {str(sample.get("output", ""))[:100]}')

## 🧱 AnyProjector Module (v0.9.7+ with Dropout + Pre-Projection)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class QFormerLayer(nn.Module):
    """Single Q-Former layer: Self-Attn → Cross-Attn → FFN.

    Supports optional dropout (v0.9.7+) for regularization.
    During inference (.eval()), dropout is automatically disabled.
    """

    def __init__(self, qformer_dim: int, encoder_dim: int, num_heads: int = 8,
                 ffn_ratio: int = 4, dropout: float = 0.0):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(
            embed_dim=qformer_dim, num_heads=num_heads, batch_first=True,
        )
        self.self_attn_norm = nn.LayerNorm(qformer_dim)
        self.self_attn_drop = nn.Dropout(dropout)

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=qformer_dim, num_heads=num_heads,
            kdim=encoder_dim, vdim=encoder_dim, batch_first=True,
        )
        self.cross_attn_norm = nn.LayerNorm(qformer_dim)
        self.cross_attn_drop = nn.Dropout(dropout)

        ffn_hidden = qformer_dim * ffn_ratio
        self.ffn = nn.Sequential(
            nn.Linear(qformer_dim, ffn_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_hidden, qformer_dim),
        )
        self.ffn_norm = nn.LayerNorm(qformer_dim)

    def forward(self, queries, encoder_out, encoder_mask=None):
        q = self.self_attn_norm(queries)
        q, _ = self.self_attn(q, q, q)
        queries = queries + self.self_attn_drop(q)

        q = self.cross_attn_norm(queries)
        q, _ = self.cross_attn(
            query=q, key=encoder_out, value=encoder_out,
            key_padding_mask=encoder_mask,
        )
        queries = queries + self.cross_attn_drop(q)

        queries = queries + self.ffn(self.ffn_norm(queries))
        return queries


class AnyProjector(nn.Module):
    """Q-Former Projector with pre-projection + dropout (v0.9.7+).

    Dùng learnable query tokens + cross-attention để nén encoder output
    thành số lượng tokens cố định. Hỗ trợ attention mask.

    Args:
        encoder_dim: Hidden size của Audio Encoder (e.g. 768, 1024).
        llm_dim: Hidden size của LLM (e.g. 1536, 3072).
        num_queries: Số learnable query tokens (output length).
        qformer_dim: Hidden dim bên trong Q-Former.
        num_layers: Số Q-Former layers.
        num_heads: Số attention heads.
        dropout: Dropout rate (0.0 = disabled).
    """

    def __init__(self, encoder_dim: int, llm_dim: int,
                 num_queries: int = 64, qformer_dim: int = 768,
                 num_layers: int = 2, num_heads: int = 8,
                 dropout: float = 0.0):
        super().__init__()
        self.encoder_dim = encoder_dim
        self.llm_dim = llm_dim
        self.num_queries = num_queries
        self.qformer_dim = qformer_dim

        self.pre_proj = nn.Sequential(
            nn.Linear(encoder_dim, encoder_dim),
            nn.GELU(),
            nn.LayerNorm(encoder_dim),
        )

        self.query_tokens = nn.Parameter(
            torch.randn(1, num_queries, qformer_dim) * 0.02
        )

        self.layers = nn.ModuleList([
            QFormerLayer(qformer_dim, encoder_dim, num_heads, dropout=dropout)
            for _ in range(num_layers)
        ])

        self.output_norm = nn.LayerNorm(qformer_dim)
        self.output_proj = nn.Sequential(
            nn.Linear(qformer_dim, llm_dim),
        )

    def forward(self, encoder_output, encoder_mask=None):
        B = encoder_output.shape[0]
        encoder_output = self.pre_proj(encoder_output)
        queries = self.query_tokens.expand(B, -1, -1)
        for layer in self.layers:
            queries = layer(queries, encoder_output, encoder_mask)
        return self.output_proj(self.output_norm(queries))

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters())

    def __repr__(self):
        n_layers = len(self.layers)
        return (
            f'AnyProjector(Q-Former)\n'
            f'  encoder_dim={self.encoder_dim}, llm_dim={self.llm_dim}\n'
            f'  pre_proj=Linear({self.encoder_dim}→{self.encoder_dim})+GELU+LN\n'
            f'  queries={self.num_queries}, qformer_dim={self.qformer_dim}\n'
            f'  layers={n_layers}, output=Linear({self.qformer_dim}→{self.llm_dim})\n'
            f'  params={self.count_parameters():,}'
        )

print('OK: AnyProjector defined (v0.9.7+ with dropout + pre-projection)')

## 📦 Imports

In [ ]:
import gc
import logging
import math
import time
import csv
import sys
import random
import shutil
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Optional

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S',
    force=True,
)
logger = logging.getLogger('phase3')
logger.setLevel(logging.INFO)

## 🔧 Config Dataclass

In [ ]:
@dataclass
class Phase3Config:
    # Models
    encoder_id: str = ENCODER_ID
    llm_id: str = LLM_ID
    dataset_id: str = DATASET_ID
    projector_ckpt: str = PROJECTOR_CKPT

    # LoRA
    lora_rank: int = LORA_RANK
    lora_alpha: int = LORA_ALPHA
    lora_dropout: float = LORA_DROPOUT
    lora_targets: list = field(default_factory=lambda: LORA_TARGETS)

    # Encoder unfreeze
    unfreeze_layers: int = UNFREEZE_LAYERS
    encoder_lr: float = ENCODER_LR

    # Training
    num_epochs: int = NUM_EPOCHS
    batch_size: int = BATCH_SIZE
    learning_rate: float = LR
    gradient_accumulation_steps: int = GRAD_ACCUM
    warmup_ratio: float = WARMUP_RATIO
    max_grad_norm: float = MAX_GRAD_NORM
    weight_decay: float = 0.01
    val_split: float = VAL_SPLIT
    max_text_tokens: int = MAX_TEXT_TOKENS
    sample_rate: int = SAMPLE_RATE
    max_audio_seconds: float = 30.0

    # Checkpointing
    save_dir: str = SAVE_DIR
    backup_dir: str = BACKUP_DIR
    save_every: int = SAVE_EVERY
    early_stopping_patience: int = PATIENCE
    early_stopping_min_delta: float = 0.001
    resume_from: Optional[str] = RESUME_FROM
    num_workers: int = NUM_WORKERS

## 📊 Dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random

class Phase3Dataset(Dataset):
    """Audio + instruction → expected output.

    Nhận list entries đã decode: [{"audio": {"array": ..., "sampling_rate": ...}, "instruction": str, "output": str}, ...]
    """

    def __init__(self, entries: list, sample_rate: int = 16000,
                 max_audio_seconds: float = 30.0):
        self.entries = entries
        self.sample_rate = sample_rate
        self.max_samples = int(max_audio_seconds * sample_rate)
        logger.info(f'Dataset: {len(entries)} samples')

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        row = self.entries[idx]
        wav = np.array(row['audio']['array'], dtype=np.float32)
        if len(wav) > self.max_samples:
            wav = wav[:self.max_samples]

        instruction = str(row.get('instruction', row.get('utt', '')))
        output = str(row.get('output', row.get('response', '')))
        return {'waveform': wav, 'instruction': instruction, 'output': output}


def collate_fn(batch):
    """Pad waveforms to same length in batch."""
    max_len = max(len(b['waveform']) for b in batch)
    waveforms = np.zeros((len(batch), max_len), dtype=np.float32)
    lengths = []
    for i, b in enumerate(batch):
        w = b['waveform']
        waveforms[i, :len(w)] = w
        lengths.append(len(w))
    return {
        'waveforms': torch.from_numpy(waveforms),
        'lengths': lengths,
        'instructions': [b['instruction'] for b in batch],
        'outputs': [b['output'] for b in batch],
    }

## 🏋️ Trainer

In [ ]:
class Phase3Trainer:
    """Trainer cho Phase 3 LoRA Tool-Calling + Encoder Unfreeze.

    Quản lý toàn bộ lifecycle: load models, train loop, checkpoint, backup.
    """

    def __init__(self, config: Phase3Config):
        self.config = config
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        logger.info(f'Device: {self.device}')

        # Models (set during setup)
        self.encoder = None
        self.projector = None
        self.llm = None
        self.tokenizer = None
        self.processor = None
        self.embed_layer = None
        self.llm_dtype = None
        self.num_queries = None

        # Training state
        self.optimizer = None
        self.scheduler = None
        self.global_step = 0
        self.start_epoch = 0
        self.best_val_loss = float('inf')
        self.patience_counter = 0
        self.log_data = []

    def setup_models(self):
        """Load tất cả models, freeze/unfreeze theo config."""
        from transformers import (
            WhisperModel, WhisperProcessor,
            AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
        )
        from peft import LoraConfig, get_peft_model
        config = self.config

        # --- 1. Whisper Encoder ---
        logger.info(f'Loading encoder: {config.encoder_id}')
        self.processor = WhisperProcessor.from_pretrained(config.encoder_id)
        self.encoder = WhisperModel.from_pretrained(config.encoder_id).encoder.to(self.device)

        for p in self.encoder.parameters():
            p.requires_grad = False

        total_layers = len(self.encoder.layers)
        if config.unfreeze_layers > 0:
            for layer in self.encoder.layers[-config.unfreeze_layers:]:
                for p in layer.parameters():
                    p.requires_grad = True
            enc_trainable = sum(p.numel() for p in self.encoder.parameters() if p.requires_grad)
            logger.info(f'  Encoder: {total_layers} layers, unfrozen last {config.unfreeze_layers} = {enc_trainable:,} params')
        else:
            self.encoder.eval()
            logger.info(f'  Encoder: frozen, {sum(p.numel() for p in self.encoder.parameters()):,} params')

        # --- 2. Projector (FROZEN) ---
        logger.info(f'Loading projector: {config.projector_ckpt}')
        ckpt = torch.load(config.projector_ckpt, map_location='cpu', weights_only=False)
        proj_config = ckpt.get('config', {})
        proj_sd = ckpt.get('projector_state_dict', ckpt)

        layer_indices = {int(k.split('.')[1]) for k in proj_sd if k.startswith('layers.')}
        num_proj_layers = max(layer_indices) + 1 if layer_indices else 4

        self.projector = AnyProjector(
            encoder_dim=proj_config.get('encoder_dim', 1024),
            llm_dim=proj_config.get('llm_dim', 1536),
            num_queries=proj_config.get('num_queries', 128),
            qformer_dim=proj_config.get('qformer_dim', 768),
            num_layers=num_proj_layers,
            num_heads=proj_config.get('qformer_heads', 16),
            dropout=proj_config.get('dropout', 0.1),
        )
        self.projector.load_state_dict(proj_sd)
        self.projector.to(self.device).eval()
        for p in self.projector.parameters():
            p.requires_grad = False
        self.num_queries = proj_config.get('num_queries', 128)
        logger.info(f'  Projector: frozen, {self.projector.count_parameters():,} params, num_queries={self.num_queries}')
        del ckpt, proj_sd

        # --- 3. LLM + LoRA ---
        logger.info(f'Loading LLM: {config.llm_id}')
        self.tokenizer = AutoTokenizer.from_pretrained(config.llm_id)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4',
        )
        self.llm = AutoModelForCausalLM.from_pretrained(
            config.llm_id, quantization_config=bnb_config,
            device_map='auto', torch_dtype=torch.bfloat16,
        )

        lora_config = LoraConfig(
            r=config.lora_rank, lora_alpha=config.lora_alpha,
            target_modules=config.lora_targets,
            lora_dropout=config.lora_dropout,
            bias='none', task_type='CAUSAL_LM',
        )
        self.llm = get_peft_model(self.llm, lora_config)
        lora_trainable = sum(p.numel() for p in self.llm.parameters() if p.requires_grad)
        logger.info(f'  LLM LoRA: {lora_trainable:,} trainable params')

        self.embed_layer = self.llm.get_base_model().get_input_embeddings()
        self.llm_dtype = torch.bfloat16

        gc.collect(); torch.cuda.empty_cache()
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / 1024**3
            reserved = torch.cuda.memory_reserved() / 1024**3
            logger.info(f'  VRAM: {allocated:.1f}GB allocated, {reserved:.1f}GB reserved')

    def setup_optimizer(self, total_steps: int):
        """Create optimizer with dual LR + warmup cosine scheduler."""
        config = self.config

        param_groups = [
            {'params': [p for p in self.llm.parameters() if p.requires_grad],
             'lr': config.learning_rate, 'name': 'lora'},
        ]
        if config.unfreeze_layers > 0:
            enc_params = [p for p in self.encoder.parameters() if p.requires_grad]
            param_groups.append({
                'params': enc_params,
                'lr': config.encoder_lr, 'name': 'encoder',
            })

        self.optimizer = torch.optim.AdamW(
            param_groups, weight_decay=config.weight_decay,
        )

        warmup_steps = int(total_steps * config.warmup_ratio)

        def lr_lambda(step):
            if step < warmup_steps:
                return float(step) / max(1, warmup_steps)
            progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
            return max(0.1, 0.5 * (1.0 + math.cos(math.pi * progress)))

        self.scheduler = torch.optim.lr_scheduler.LambdaLR(self.optimizer, lr_lambda)
        logger.info(f'Optimizer: AdamW, warmup={warmup_steps}/{total_steps}')
        for g in param_groups:
            logger.info(f'  {g.get("name", "?")}: lr={g["lr"]}, {sum(p.numel() for p in g["params"]):,} params')

    def _get_all_trainable(self):
        """Collect all trainable params for grad clipping."""
        params = [p for p in self.llm.parameters() if p.requires_grad]
        if self.config.unfreeze_layers > 0:
            params += [p for p in self.encoder.parameters() if p.requires_grad]
        return params

    def _vram_info(self) -> str:
        if torch.cuda.is_available():
            alloc = torch.cuda.memory_allocated() / 1024**3
            total = torch.cuda.get_device_properties(0).total_memory / 1024**3
            return f'{alloc:.1f}/{total:.1f}GB'
        return 'N/A'

    def _encode_audio(self, waveforms, lengths):
        """Audio → Whisper → Projector → audio_embeds (B, nq, llm_dim).

        NOTE: Projector forward is INSIDE grad context so gradients
        flow THROUGH frozen projector back to unfrozen encoder layers.
        Projector params have requires_grad=False so they won't update,
        but the computation graph is preserved for encoder gradients.
        """
        inputs = self.processor(
            [w.numpy() for w in waveforms],
            sampling_rate=self.config.sample_rate,
            return_tensors='pt', padding='max_length',
        )
        input_features = inputs.input_features.to(self.device)

        with torch.set_grad_enabled(self.config.unfreeze_layers > 0):
            enc_out = self.encoder(input_features).last_hidden_state
            # Projector inside grad context: graph preserved for encoder backprop
            audio_embeds = self.projector(enc_out.float())
        return audio_embeds

    def _build_inputs(self, audio_embeds, instructions, targets):
        """Build [instruction | audio | suffix | target] embeddings + labels.

        Mỗi sample dùng instruction riêng từ dataset, không hardcode.
        Structure:
            <|im_start|>user
            {instruction}
            [audio tokens]<|im_end|>
            <|im_start|>assistant
            {target}<|im_end|>
        """
        B = audio_embeds.shape[0]
        nq = audio_embeds.shape[1]

        # Prefix: user turn + instruction (mỗi sample instruction riêng)
        prefix_texts = [f'<|im_start|>user\n{inst}\n' for inst in instructions]
        self.tokenizer.padding_side = 'left'
        prefix_tok = self.tokenizer(
            prefix_texts, return_tensors='pt', padding=True,
            add_special_tokens=False,
        ).to(self.device)
        with torch.no_grad():
            prefix_embeds = self.embed_layer(prefix_tok.input_ids)
        prefix_mask = prefix_tok.attention_mask

        # Audio mask
        audio_mask = torch.ones((B, nq), dtype=torch.long, device=self.device)

        # Suffix: chuyển sang assistant turn
        suffix_text = '<|im_end|>\n<|im_start|>assistant\n'
        self.tokenizer.padding_side = 'right'
        suffix_tok = self.tokenizer(
            [suffix_text] * B, return_tensors='pt', padding=True,
            add_special_tokens=False,
        ).to(self.device)
        with torch.no_grad():
            suffix_embeds = self.embed_layer(suffix_tok.input_ids)
        suffix_mask = suffix_tok.attention_mask

        # Target
        target_texts = [f'{t}<|im_end|>' for t in targets]
        target_tok = self.tokenizer(
            target_texts, return_tensors='pt', padding=True,
            add_special_tokens=False, truncation=True,
            max_length=self.config.max_text_tokens,
        ).to(self.device)
        with torch.no_grad():
            target_embeds = self.embed_layer(target_tok.input_ids)
        target_mask = target_tok.attention_mask

        # Concat all
        full_embeds = torch.cat(
            [prefix_embeds, audio_embeds, suffix_embeds, target_embeds], dim=1
        ).to(self.llm_dtype)
        full_mask = torch.cat(
            [prefix_mask, audio_mask, suffix_mask, target_mask], dim=1
        )

        # Labels: -100 everywhere except target tokens
        ignore_prefix = torch.full_like(prefix_tok.input_ids, -100)
        ignore_audio = torch.full((B, nq), -100, dtype=torch.long, device=self.device)
        ignore_suffix = torch.full_like(suffix_tok.input_ids, -100)
        target_labels = target_tok.input_ids.clone()
        target_labels[target_mask == 0] = -100
        labels = torch.cat(
            [ignore_prefix, ignore_audio, ignore_suffix, target_labels], dim=1
        )

        return full_embeds, full_mask, labels

    def process_batch(self, batch):
        """Process 1 batch → return loss."""
        audio_embeds = self._encode_audio(batch['waveforms'], batch['lengths'])
        full_embeds, full_mask, labels = self._build_inputs(
            audio_embeds, batch['instructions'], batch['outputs'],
        )
        outputs = self.llm(
            inputs_embeds=full_embeds, attention_mask=full_mask, labels=labels,
        )
        return outputs.loss

    def save_checkpoint(self, tag, val_loss=None):
        """Save LoRA + encoder weights + training state."""
        config = self.config
        save_dir = Path(config.save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)

        ckpt_dir = save_dir / str(tag)
        ckpt_dir.mkdir(parents=True, exist_ok=True)

        self.llm.save_pretrained(str(ckpt_dir))

        if config.unfreeze_layers > 0:
            torch.save(self.encoder.state_dict(), str(ckpt_dir / 'encoder.pt'))

        torch.save({
            'global_step': self.global_step,
            'epoch': tag if isinstance(tag, int) else -1,
            'val_loss': val_loss,
            'num_queries': self.num_queries,
            'unfreeze_layers': config.unfreeze_layers,
            'config': {
                'encoder_id': config.encoder_id,
                'llm_id': config.llm_id,
                'lora_rank': config.lora_rank,
                'lora_alpha': config.lora_alpha,
            },
        }, str(ckpt_dir / 'meta.pt'))
        logger.info(f'  💾 Saved: {ckpt_dir}')

        try:
            drive_dir = Path(config.backup_dir) / str(tag)
            if drive_dir.exists():
                shutil.rmtree(drive_dir)
            shutil.copytree(ckpt_dir, drive_dir)
            logger.info(f'  → Drive: {tag}')
        except Exception as e:
            logger.warning(f'  Drive backup failed: {e}')

    def _backup_to_drive(self):
        """Auto-backup training log to Google Drive."""
        config = self.config
        log_path = Path(config.save_dir) / 'training_log.csv'
        if log_path.exists():
            try:
                dst = Path(config.backup_dir) / 'training_log.csv'
                shutil.copy2(log_path, dst)
                logger.info(f'📊 Log backed up to Drive')
            except Exception as e:
                logger.warning(f'Log backup failed: {e}')

    def train(self):
        """Full training loop with verbose logging."""
        config = self.config

        # --- Setup Models ---
        self.setup_models()

        # --- Dataset: gộp tất cả splits, decode audio, chia 80/20 ---
        all_entries = []
        for split_name in raw_ds:
            logger.info(f'Loading split "{split_name}": {len(raw_ds[split_name])} samples')
            all_entries.extend(list(raw_ds[split_name]))
        logger.info(f'Total: {len(all_entries)} samples')

        random.seed(42)
        random.shuffle(all_entries)
        split_idx = int(len(all_entries) * (1 - config.val_split))
        train_entries = all_entries[:split_idx]
        val_entries = all_entries[split_idx:]

        train_dataset = Phase3Dataset(train_entries, config.sample_rate, config.max_audio_seconds)
        val_dataset = Phase3Dataset(val_entries, config.sample_rate, config.max_audio_seconds)

        nw = config.num_workers
        train_loader = DataLoader(
            train_dataset, batch_size=config.batch_size,
            shuffle=True, collate_fn=collate_fn,
            num_workers=nw, pin_memory=True,
        )
        val_loader = DataLoader(
            val_dataset, batch_size=config.batch_size,
            shuffle=False, collate_fn=collate_fn,
            num_workers=nw, pin_memory=True,
        )

        logger.info(f'Dataset split: train={len(train_dataset)}, val={len(val_dataset)}')
        logger.info(f'Batches/epoch: {len(train_loader)}')

        # --- Optimizer ---
        steps_per_epoch = math.ceil(len(train_loader) / config.gradient_accumulation_steps)
        total_steps = steps_per_epoch * config.num_epochs
        self.setup_optimizer(total_steps)

        all_trainable = self._get_all_trainable()
        total_trainable = sum(p.numel() for p in all_trainable)

        # --- Banner ---
        logger.info('')
        logger.info('╔' + '═' * 58 + '╗')
        logger.info('║  🚀 PHASE 3 — Speech Understanding + Tool Calling       ║')
        logger.info('╠' + '═' * 58 + '╣')
        logger.info(f'║  Encoder:    {config.encoder_id:<43}║')
        logger.info(f'║  LLM:        {config.llm_id:<43}║')
        pinfo = f'{self.num_queries} tokens (frozen, {self.projector.count_parameters():,} params)'
        logger.info(f'║  Projector:  {pinfo:<43}║')
        einfo = f'unfreeze last {config.unfreeze_layers} layer @ LR={config.encoder_lr}'
        logger.info(f'║  Encoder:    {einfo:<43}║')
        linfo = f'r={config.lora_rank}, α={config.lora_alpha}, targets={config.lora_targets}'
        logger.info(f'║  LoRA:       {linfo:<43}║')
        logger.info(f'║  Trainable:  {total_trainable:>10,} params{" "*30}║')
        logger.info(f'║  Epochs:     {config.num_epochs:<43}║')
        eff_batch = config.batch_size * config.gradient_accumulation_steps
        logger.info(f'║  Batch:      {config.batch_size} × {config.gradient_accumulation_steps} = {eff_batch} effective{" "*28}║')
        logger.info(f'║  VRAM:       {self._vram_info():<43}║')
        logger.info('╚' + '═' * 58 + '╝')
        logger.info('')

        # --- Training Loop ---
        history = []

        self.llm.train()
        if config.unfreeze_layers > 0:
            self.encoder.train()

        for epoch in range(1, config.num_epochs + 1):
            # ============ TRAIN ============
            logger.info(f'━━━ Epoch {epoch}/{config.num_epochs} ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
            self.llm.train()
            if config.unfreeze_layers > 0:
                self.encoder.train()

            train_loss_sum = 0.0
            train_steps = 0
            self.optimizer.zero_grad()
            epoch_start = time.time()

            for batch_idx, batch in enumerate(train_loader):
                loss = self.process_batch(batch)
                loss_val = loss.float()
                scaled_loss = loss_val / config.gradient_accumulation_steps
                scaled_loss.backward()

                batch_loss = loss_val.item()
                train_loss_sum += batch_loss
                train_steps += 1

                # Progress bar
                total = len(train_loader)
                done = batch_idx + 1
                pct = done / total
                filled = int(25 * pct)
                bar = '█' * filled + '░' * (25 - filled)
                avg_loss = train_loss_sum / train_steps
                lr = self.optimizer.param_groups[0]['lr']
                elapsed_s = time.time() - epoch_start
                print(
                    f'\r  {bar} {done}/{total} | '
                    f'loss={avg_loss:.4f} | lr={lr:.2e} | '
                    f'{elapsed_s:.0f}s | VRAM {self._vram_info()}',
                    end='', flush=True,
                )

                if (batch_idx + 1) % config.gradient_accumulation_steps == 0:
                    torch.nn.utils.clip_grad_norm_(all_trainable, config.max_grad_norm)
                    self.optimizer.step()
                    self.scheduler.step()
                    self.optimizer.zero_grad()
                    self.global_step += 1

            # Flush remaining grads
            if train_steps % config.gradient_accumulation_steps != 0:
                torch.nn.utils.clip_grad_norm_(all_trainable, config.max_grad_norm)
                self.optimizer.step()
                self.scheduler.step()
                self.optimizer.zero_grad()
                self.global_step += 1

            train_avg = train_loss_sum / max(train_steps, 1)
            elapsed = time.time() - epoch_start
            print()  # Newline after progress bar

            # ============ VALIDATION ============
            logger.info(f'  📊 Validating...')
            self.llm.eval()
            if config.unfreeze_layers > 0:
                self.encoder.eval()

            val_loss_sum = 0.0
            val_steps = 0
            with torch.no_grad():
                for val_idx, batch in enumerate(val_loader):
                    loss = self.process_batch(batch)
                    val_loss = loss.float().item()
                    val_loss_sum += val_loss
                    val_steps += 1

            val_avg = val_loss_sum / max(val_steps, 1)

            # ============ EPOCH SUMMARY ============
            history.append({'epoch': epoch, 'train': train_avg, 'val': val_avg})
            overfit_ratio = val_avg / max(train_avg, 1e-8)

            # Early stopping (with min_delta)
            if val_avg < self.best_val_loss - config.early_stopping_min_delta:
                self.best_val_loss = val_avg
                self.patience_counter = 0
                improved = '★ BEST'
                self.save_checkpoint('best', val_avg)
            else:
                self.patience_counter += 1
                improved = f'wait {self.patience_counter}/{config.early_stopping_patience}'

            # Overfit warning
            if overfit_ratio > 2.0: fit_status = '⚠️ OVERFIT'
            elif overfit_ratio > 1.5: fit_status = '😐 MILD'
            elif overfit_ratio > 1.2: fit_status = '👍 OK'
            else: fit_status = '✅ GOOD'

            logger.info('')
            logger.info(f'  ┌─────────────────────────────────────────────┐')
            logger.info(f'  │ Epoch {epoch:>2}/{config.num_epochs} Summary{" " * 27}│')
            logger.info(f'  ├─────────────────────────────────────────────┤')
            logger.info(f'  │ Train Loss:    {train_avg:>8.4f}                      │')
            logger.info(f'  │ Val Loss:      {val_avg:>8.4f}  {improved:<19}│')
            logger.info(f'  │ Overfit Ratio: {overfit_ratio:>8.2f}x {fit_status:<18}│')
            logger.info(f'  │ Best Val:      {self.best_val_loss:>8.4f}                      │')
            logger.info(f'  │ LR (LoRA):     {self.optimizer.param_groups[0]["lr"]:>8.2e}                      │')
            if config.unfreeze_layers > 0:
                logger.info(f'  │ LR (Encoder):  {self.optimizer.param_groups[1]["lr"]:>8.2e}                      │')
            logger.info(f'  │ Time:          {elapsed:>8.1f}s                     │')
            logger.info(f'  │ Global Step:   {self.global_step:>8}                      │')
            logger.info(f'  │ VRAM:          {self._vram_info():>12}                  │')
            logger.info(f'  └─────────────────────────────────────────────┘')
            logger.info('')

            self.log_data.append({
                'epoch': epoch, 'train_loss': round(train_avg, 5),
                'val_loss': round(val_avg, 5),
                'lr_lora': self.optimizer.param_groups[0]['lr'],
                'lr_enc': self.optimizer.param_groups[1]['lr'] if config.unfreeze_layers > 0 else 0,
                'overfit_ratio': round(overfit_ratio, 3),
                'elapsed_s': round(elapsed, 1),
            })

            # Save periodic
            if epoch % config.save_every == 0:
                self.save_checkpoint(epoch, val_avg)

            # Early stopping check
            if self.patience_counter >= config.early_stopping_patience:
                logger.info(f'🛑 Early stopping! Val loss không cải thiện {config.early_stopping_min_delta}+ sau {config.early_stopping_patience} epochs.')
                logger.info(f'   Best model: val_loss={self.best_val_loss:.4f}')
                self.save_checkpoint(f'early_stop_e{epoch}', val_avg)
                break

        # Save final
        self.save_checkpoint('final', val_avg)

        # Save training log CSV
        if self.log_data:
            log_path = Path(config.save_dir) / 'training_log.csv'
            with open(log_path, 'w', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=self.log_data[0].keys())
                writer.writeheader()
                writer.writerows(self.log_data)
            logger.info(f'📊 Training log: {log_path}')

        # ============ FINAL SUMMARY ============
        logger.info('')
        logger.info('╔' + '═' * 58 + '╗')
        logger.info('║         🏁 TRAINING COMPLETE                             ║')
        logger.info('╠' + '═' * 58 + '╣')
        logger.info(f'║  Best Val Loss:  {self.best_val_loss:<39.4f}║')
        logger.info(f'║  Final Train:    {history[-1]["train"]:<39.4f}║')
        logger.info(f'║  Total Steps:    {self.global_step:<39}║')
        logger.info(f'║  Checkpoints:    {config.save_dir:<39}║')
        logger.info('╠' + '═' * 58 + '╣')
        logger.info('║  Loss History (last 10 epochs):                          ║')
        for h in history[-10:]:
            bar_len = int(max(0, min(30, (h['train'] / max(history[0]['train'], 1)) * 30)))
            bar = '█' * bar_len + '░' * (30 - bar_len)
            logger.info(f'║  E{h["epoch"]:>2} T={h["train"]:.3f} V={h["val"]:.3f} {bar} ║')
        logger.info('╚' + '═' * 58 + '╝')

        self._backup_to_drive()

print('OK: Phase3Trainer defined')

## 🚀 Run Training

In [ ]:
config = Phase3Config()
print(config)
print()
trainer = Phase3Trainer(config)
trainer.train()

## 🔍 Quick Eval — Generate from Audio

In [ ]:
@torch.no_grad()
def generate_from_audio(idx, max_tokens=128):
    trainer.llm.eval()
    trainer.encoder.eval()

    # Load a val sample
    if 'test' in raw_ds:
        sample = raw_ds['test'][idx]
    else:
        split = list(raw_ds.keys())[0]
        sample = raw_ds[split][-(idx+1)]  # from end = val

    wav = np.array(sample['audio']['array'], dtype=np.float32)
    instruction = str(sample.get('instruction', 'Phản hồi câu nói dưới dạng tool call hoặc trả lời tự nhiên.'))

    inputs = trainer.processor([wav], sampling_rate=SAMPLE_RATE, return_tensors='pt', padding='max_length')
    enc_out = trainer.encoder(inputs.input_features.to(trainer.device)).last_hidden_state
    audio_embeds = trainer.projector(enc_out.float())

    prefix = f'<|im_start|>user\n{instruction}\n'
    suffix = '<|im_end|>\n<|im_start|>assistant\n'
    prefix_ids = trainer.tokenizer(prefix, return_tensors='pt', add_special_tokens=False).input_ids.to(trainer.device)
    suffix_ids = trainer.tokenizer(suffix, return_tensors='pt', add_special_tokens=False).input_ids.to(trainer.device)
    prefix_embeds = trainer.embed_layer(prefix_ids)
    suffix_embeds = trainer.embed_layer(suffix_ids)

    input_embeds = torch.cat([prefix_embeds, audio_embeds, suffix_embeds], dim=1).to(trainer.llm_dtype)
    attn_mask = torch.ones(1, input_embeds.shape[1], dtype=torch.long, device=trainer.device)

    outputs = trainer.llm.generate(
        inputs_embeds=input_embeds, attention_mask=attn_mask,
        max_new_tokens=max_tokens, do_sample=False,
        eos_token_id=trainer.tokenizer.eos_token_id,
        pad_token_id=trainer.tokenizer.pad_token_id,
    )
    generated = trainer.tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f'--- Sample {idx} ---')
    print(f'Instruction: {instruction[:100]}')
    print(f'Expected:    {str(sample.get("output", ""))[:200]}')
    print(f'Generated:   {generated[:200]}')
    print()

# Test on 5 samples
for i in range(5):
    generate_from_audio(i)

## 💾 Download Best Checkpoint

In [ ]:
from google.colab import files
import zipfile

best_dir = f'{SAVE_DIR}/best'
zip_path = f'{SAVE_DIR}/best_checkpoint.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk(best_dir):
        for fname in fnames:
            fpath = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, best_dir)
            zf.write(fpath, arcname)
print(f'Zipped: {zip_path}')
files.download(zip_path)